In [ ]:
def main(datasource, start_date, end_date):
    import numpy as np
    import pandas as pd
    import dai

    if isinstance(datasource, dict):
        bar1m = datasource.get(
            "bar1m",
            datasource.get("bigalpha_2026_stock_bar1m", "bigalpha_2026_stock_bar1m"),
        )
    else:
        bar1m = datasource or "bigalpha_2026_stock_bar1m"

    sql = f"""
    WITH raw AS (
      SELECT
        date::DATE::DATETIME AS trading_day,
        instrument::string AS instrument,
        date AS minute_time,
        EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) AS hhmm,
        open,
        high,
        low,
        close,
        volume,
        amount,
        deal_number,
        ask_price1,
        ask_price2,
        ask_price3,
        ask_price4,
        ask_price5,
        bid_price1,
        bid_price2,
        bid_price3,
        bid_price4,
        bid_price5,
        ask_volume1,
        ask_volume2,
        ask_volume3,
        ask_volume4,
        ask_volume5,
        bid_volume1,
        bid_volume2,
        bid_volume3,
        bid_volume4,
        bid_volume5,
        ask_num_orders1,
        ask_num_orders2,
        ask_num_orders3,
        ask_num_orders4,
        ask_num_orders5,
        bid_num_orders1,
        bid_num_orders2,
        bid_num_orders3,
        bid_num_orders4,
        bid_num_orders5
      FROM {bar1m}
    ),
    primitive AS (
      SELECT
        trading_day,
        instrument,
        hhmm,
        LEAST(GREATEST((COALESCE(high, 0) - COALESCE(low, 0)) / NULLIF(ABS(((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5)), 0), 0), 0.2) AS p0,
        LEAST(GREATEST((COALESCE(close, 0) - COALESCE(open, 0)) / NULLIF(ABS(open), 0), -0.2), 0.2) AS p1,
        LEAST(GREATEST(((((COALESCE(ask_price1,0)*COALESCE(ask_volume1,0) + COALESCE(bid_price1,0)*COALESCE(bid_volume1,0) + COALESCE(ask_price2,0)*COALESCE(ask_volume2,0) + COALESCE(bid_price2,0)*COALESCE(bid_volume2,0) + COALESCE(ask_price3,0)*COALESCE(ask_volume3,0) + COALESCE(bid_price3,0)*COALESCE(bid_volume3,0) + COALESCE(ask_price4,0)*COALESCE(ask_volume4,0) + COALESCE(bid_price4,0)*COALESCE(bid_volume4,0) + COALESCE(ask_price5,0)*COALESCE(ask_volume5,0) + COALESCE(bid_price5,0)*COALESCE(bid_volume5,0)) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)), 0)) - ((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5)) / NULLIF(ABS(((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5)), 0)), -0.2), 0.2) AS p2,
        LEAST(GREATEST((((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5) - ((COALESCE(ask_price1,0)*COALESCE(bid_volume1,0) + COALESCE(bid_price1,0)*COALESCE(ask_volume1,0)) / NULLIF(COALESCE(ask_volume1,0)+COALESCE(bid_volume1,0),0))) / NULLIF(ABS((COALESCE(ask_price1, 0) - COALESCE(bid_price1, 0))), 0), -2), 2) AS p3,
        (((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0)) - (COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0))) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)), 0)) * LN(1 + GREATEST(COALESCE(amount, 0), 0)) AS p4
      FROM raw
    ),
    daily AS (
      SELECT
        trading_day AS date,
        instrument,
        ROUND(CAST((STDDEV_SAMP(p0)) AS DOUBLE), 8) AS f0,
        ROUND(CAST((QUANTILE_CONT(p0, 0.90)) AS DOUBLE), 8) AS f1,
        ROUND(CAST((AVG(p0)) AS DOUBLE), 8) AS f2,
        ROUND(CAST((QUANTILE_CONT(p1, 0.90)) AS DOUBLE), 8) AS f3,
        ROUND(CAST((STDDEV_SAMP(p2)) AS DOUBLE), 8) AS f4,
        ROUND(CAST((QUANTILE_CONT(p3, 0.10)) AS DOUBLE), 8) AS f5,
        ROUND(CAST((AVG(p2)) AS DOUBLE), 8) AS f6,
        ROUND(CAST((STDDEV_SAMP(p1)) AS DOUBLE), 8) AS f7,
        ROUND(CAST((QUANTILE_CONT(p4, 0.90)) AS DOUBLE), 8) AS f8,
        ROUND(CAST((QUANTILE_CONT(p4, 0.10)) AS DOUBLE), 8) AS f9,
        ROUND(CAST((AVG(p3)) AS DOUBLE), 8) AS f10
      FROM primitive
      GROUP BY trading_day, instrument
    )
    SELECT date, instrument, f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10
    FROM daily
    """
    data = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    if data.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    raw_columns = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10']
    data = data[["date", "instrument", *raw_columns]].copy()
    data["date"] = pd.to_datetime(data["date"]).dt.normalize()
    data["instrument"] = data["instrument"].astype(str)
    for column in raw_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    data = data.replace([np.inf, -np.inf], np.nan)

    def cs_rank(series):
        values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
        values = values.fillna(values.median())
        return values.rank(method="average", pct=True).fillna(0.5) * 2.0 - 1.0

    def rank_factor(series):
        return pd.Series(series, index=data.index).groupby(
            data["date"], group_keys=False
        ).transform(cs_rank)

    ranks = data.groupby("date", group_keys=False)[raw_columns].transform(cs_rank)
    model_x = ranks[raw_columns].to_numpy(dtype=np.float32, copy=True)
    model_mean = model_x.mean(axis=1, keepdims=True, dtype=np.float32)
    model_var = ((model_x - model_mean) ** 2).mean(axis=1, keepdims=True, dtype=np.float32)
    model_z = (model_x - model_mean) / np.sqrt(model_var + np.float32(1e-5))
    model_z = model_z * np.asarray([0.84533924,0.8642015 ,0.9542706 ,0.8119532 ,0.91597843,0.85900956,0.82874703,1.0031822 ,0.87961364,0.96841055,0.86687195], dtype=np.float32).reshape((11,)) + np.asarray([ 0.26623735 , 0.2388632  ,-0.23455028 ,-0.26695552 ,-0.019216677,-0.26344797 ,-0.25782    , 0.08739834 , 0.2541129  , 0.31475174 ,
      0.25424257 ], dtype=np.float32).reshape((11,))
    fm_left = np.tanh(model_z @ np.asarray([-5.66923693e-02,-2.31475815e-01, 1.94051247e-02,-1.43172190e-01, 2.68329114e-01,-5.65767214e-02,-5.32125756e-02,-6.33073002e-02,
     -3.07165921e-01, 4.00825143e-01,-2.13908609e-02,-1.49919674e-01,-1.35479599e-01, 2.87939161e-01, 2.32073098e-01, 5.92292333e-03,
      2.05813527e-01, 9.03181061e-02,-1.54699504e-01,-3.32082927e-01,-3.35113443e-02,-1.07102536e-01, 4.30135317e-02, 1.24253757e-01,
      6.45937100e-02,-1.03035502e-01,-1.57686502e-01, 4.04351056e-01, 1.86965704e-01,-1.44718722e-01, 1.25880063e-01, 2.14975938e-01,
      6.78881630e-02, 3.53969721e-04, 3.61724943e-01,-3.25169899e-02,-1.30110383e-01,-3.12967360e-01,-1.15989648e-01,-2.55733997e-01,
      1.06806874e-01, 1.40560523e-01, 7.23747462e-02,-1.58389196e-01,-2.79631466e-02, 2.66072929e-01, 1.90790221e-01,-4.40070659e-01,
      1.31513417e-01, 4.46759760e-02,-5.62015772e-02, 3.05064827e-01, 2.54876047e-01, 5.82572781e-02,-1.20575562e-01,-1.15960211e-01,
      3.73734757e-02,-2.28610396e-01,-1.80125222e-01,-1.73044562e-01,-1.70129970e-01,-1.96229172e-04, 2.30050325e-01,-2.44314238e-01,
      1.44402951e-01, 2.80467123e-02,-3.36677432e-02, 6.26612781e-03,-2.48797797e-03,-1.51449636e-01,-1.16890900e-01,-1.21413574e-01,
     -8.63456428e-02, 1.55927446e-02, 3.39917481e-01, 1.92815781e-01,-6.46068901e-02, 3.88869122e-02,-1.56224340e-01, 1.43449917e-01,
      2.56320685e-01, 7.51436949e-02, 2.33719960e-01, 3.41644228e-01, 1.96726725e-01,-1.31112605e-01,-3.70007068e-01,-2.27174252e-01], dtype=np.float32).reshape((8, 11)).T)
    fm_right = np.tanh(model_z @ np.asarray([ 0.13218595  , 0.25804526  ,-0.15374391  , 0.3618168   , 0.108967476 ,-0.10821828  ,-0.057149775 ,-0.12324947  ,-0.15600182  ,
      0.026354903 , 0.3393539   ,-0.008274573 , 0.032387987 ,-0.24755473  ,-0.1944028   , 0.08484719  , 0.04719364  ,-0.07563108  ,
     -0.17472325  ,-0.24586765  , 0.22851005  ,-0.016987866 ,-0.036076702 ,-0.12358366  ,-0.17295235  ,-0.07579875  ,-0.11130106  ,
     -0.3978078   ,-0.20481758  ,-0.09928147  ,-0.23418505  ,-0.089649364 ,-0.035878398 ,-0.22313724  , 0.0073458543,-0.057432927 ,
      0.14278556  , 0.27461636  ,-0.20032169  ,-0.194384    , 0.18734854  , 0.16477038  , 0.3562602   ,-0.11375911  , 0.4147034   ,
      0.16534266  ,-0.1688378   ,-0.4244292   ,-0.13523419  ,-0.15132447  ,-0.25355795  ,-0.15665653  , 0.15884742  , 0.26743835  ,
      0.33122766  ,-0.04034622  , 0.23154418  , 0.05675525  , 0.02455388  ,-0.24447343  ,-0.10057999  , 0.24004897  ,-0.055857506 ,
     -0.027086243 ,-0.092279635 ,-0.37734336  , 0.16593121  , 0.12457796  , 0.18406321  ,-0.17791867  ,-0.18761113  ,-0.0105112605,
     -0.2127647   ,-0.13153112  ,-0.12662669  , 0.07351735  , 0.101132974 ,-0.27300736  ,-0.09227536  ,-0.24580552  ,-0.01974962  ,
      0.03529101  , 0.0049385666, 0.22361419  ,-0.096723415 ,-0.22412269  , 0.019023096 ,-0.0114610195], dtype=np.float32).reshape((8, 11)).T)
    fm_linear = model_z @ np.asarray([-0.028753446,-0.028617343,-0.00679231 , 0.09451338 , 0.022933796, 0.018257855, 0.06550327 ,-0.123829335,-0.093434885,-0.05906535 ,
      0.017052209], dtype=np.float32).reshape((1, 11)).T + np.asarray([0.110224165], dtype=np.float32).reshape((1,))
    raw_factor = (fm_linear + (fm_left * fm_right) @ np.asarray([ 0.059794646,-0.15949932 , 0.13409804 , 0.07835932 , 0.1704812  ,-0.18036316 , 0.27749038 , 0.15138006 ], dtype=np.float32).reshape((1, 8)).T).ravel()
    data["factor"] = pd.Series(raw_factor, index=data.index, dtype="float64")
    data["factor"] = pd.to_numeric(data["factor"], errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    daily_median = data.groupby("date")["factor"].transform("median")
    data["factor"] = data["factor"].fillna(daily_median).fillna(0.0)
    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    data = data[(data["date"] >= start_ts) & (data["date"] <= end_ts)]
    return data[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)
